## 4. Load

Agrega `matches_transformed` por temporada, pais y liga
y escribe KPIs en `football_dev.gold.season_stats`.


In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "gold")


In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")


In [0]:
df_silver = spark.table(f"{catalogo}.{esquema_source}.matches_transformed")


In [0]:
df_gold = df_silver.groupBy("season_year", "country", "league_name").agg(
    count(col("match_id")).alias("conteo"),
    sum(col("total_goals")).alias("total_goals"),
    max(col("total_goals")).alias("max_goals"),
    min(col("total_goals")).alias("min_goals"),
    sum(when(col("is_classic") == "Clasico", 1).otherwise(0)).alias("classic_count"),
    sum(when(col("result_type") == "Local", 1).otherwise(0)).alias("home_win_count")
).orderBy(col("season_year").desc(), col("country"))


In [0]:
df_gold.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.season_stats")
